In [ ]:
DF_TRAIN = 'train_prep.csv'

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, f1_score, precision_score

DF_TEST = 'test.csv'
DF_SUB = 'gender_submission.csv'

df_train = pd.read_csv(DF_TRAIN)
X_train = df_train.drop('Survived', axis=1)
Y_train = df_train['Survived']

df_test = pd.read_csv(DF_TEST)
# Y_test = pd.read_csv(DF_SUB)['Survived'].values()


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

CAT_FEATURES = ['Pclass', 'Sex', 'Designation']


def evaluate_model(model, X, y, cv, scoring_dict, n_jobs=-1):
    """
    Универсальная функция для cross-validation
    """

    results = {}
    for metric_name, scorer in scoring_dict.items():
        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring=scorer,
            n_jobs=n_jobs
        )
        results[metric_name] = float(np.mean(scores))
    return results


"""
Словарь метрик для оценки модели
"""
score = {
    'Accuracy': 'accuracy',
    'F1': make_scorer(f1_score, average='binary'),
    'ROC-AUC': 'roc_auc',
    'Precision': make_scorer(precision_score, average='binary'),
}

In [ ]:
results_all = {}

In [4]:
from core import (
    skf,
    X_train,
    Y_train,
    evaluate_model,
    CAT_FEATURES,
    score
)

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV


cat = CatBoostClassifier(
    random_state=42,
    verbose=0,
    thread_count=-1,
    eval_metric='AUC',
    od_type='Iter',
    od_wait=50
)

param_dist_cat = {
    'iterations': [200, 500, 800, 1000, 1500],
    'depth': [1, 2, 3, 4, 5, 6],
    'learning_rate': [0.002, 0.005, 0.008, 0.01, 0.02],
    'l2_leaf_reg': [1, 3, 5, 7],
    'bagging_temperature': [0, 0.5, 1, 1.5, 2],
    'random_strength': [3, 5, 8],
    'border_count': [32, 64, 128],
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'min_data_in_leaf': [1, 3, 5, 10],
}

random_cat = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist_cat,
    n_iter=200,
    cv=skf,
    scoring='roc_auc',
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_cat.fit(X_train, Y_train, cat_features=CAT_FEATURES)

best_cat = random_cat.best_estimator_

print(f'Лучшие параметры CatBoost: {random_cat.best_params_}')
print(f'Лучшая ROC_AUC: {random_cat.best_score_:.4f}')

results_cat = evaluate_model(
    model=best_cat,
    X=X_train,
    y=Y_train,
    cv=skf,
    scoring_dict=score 
)

results_all = {'CatBoost': results_cat}

Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Лучшие параметры CatBoost: {'random_strength': 5, 'min_data_in_leaf': 1, 'learning_rate': 0.02, 'l2_leaf_reg': 7, 'iterations': 800, 'grow_policy': 'Lossguide', 'depth': 4, 'border_count': 32, 'bagging_temperature': 2}
Лучшая ROC‑AUC: 0.8833


In [5]:
results_all

{'CatBoost': {'Accuracy': 0.8484903646977591,
  'F1': 0.7905678961387089,
  'ROC-AUC': 0.8825451409849215,
  'Precision': 0.8420647699494351}}